# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassaan-Raza/FlyRank-Intership/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I'm choosing Lane 2: Refresh / Content Opportunity Scoring. This lane produces a ranked, actionable output, a review queue with reason codes, rather than just an exploratory report, and it closely matches how I've approached prior projects: scoring and ranking candidates with transparent, inspectable reasons rather than a single opaque number. I want to spend my effort this cycle on getting the ML fundamentals right (leakage discipline, validation design, honest metrics) rather than on inventing the problem framing from scratch.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**Question:** Which content pages should a reviewer look at first for refresh, expansion, protection, pruning, or monitoring, given limited review capacity?

**Unit of analysis:** one content item (content_hash_id).

**Decision this improves:** how a reviewer with limited time allocates attention across a large content inventory.

**Action someone takes:** a human reviewer manually inspects the flagged page and decides whether to refresh, expand, protect, prune, or monitor it. The model doesn't act on its own, it prioritizes.

**Cost of a wrong call, both directions:**

*   **False positive (flagging a healthy page):** wastes a reviewer's limited time on a page that didn't need attention.
*   **False negative (missing a genuinely declining page):** a real opportunity or problem goes unaddressed until it's found some other way, which becomes more costly the longer it's missed.

**Why data/ML helps here:** with a large inventory and limited review capacity, a transparent ranking beats either reviewing pages randomly or relying only on a fixed rule, as long as the ranking is validated honestly and its reason codes stay inspectable.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

The starter dataset has 30,000 rows total, and all 30,000 survive the filter (impressions_90d > 0 and content_age_days >= 90), meaning every row in this starter slice already meets the basic visibility and maturity bar.

Of those, 16,262 rows (54.2%) have trend_direction == "down", which is a fairly balanced proxy label, not a rare-event problem, so standard classification metrics should behave reasonably here.

Applying the declining_with_demand reason code (trend_direction == "down" and impressions_90d >= 100) captures 13,152 rows (43.8% of the filtered set), showing that most declining pages also have real demand behind them, worth reviewing. By contrast, the stale_visible_page reason code (days_since_last_update >= 180 and impressions_90d >= 500) only catches 17 rows (0.1%), suggesting the starter slice either has very little visible staleness or a high impressions_90d >= 500 bar most rows don't clear. This gap between reason codes is itself useful evidence: a single fixed rule like stale_visible_page would surface almost nothing here, which is part of the argument for a learned ranking that can weigh multiple weak signals together instead of relying on one rule.

In [11]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

total_rows = len(df)
print(f"Total rows in raw dataset: {total_rows:,}")

filtered = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
filtered_rows = len(filtered)
print(f"Rows after filter (impressions_90d > 0 and content_age_days >= 90): {filtered_rows:,}")

declining_count = (filtered["trend_direction"] == "down").sum()
declining_pct = declining_count / filtered_rows
print(f"\nRows with trend_direction == 'down': {declining_count:,} ({declining_pct:.1%})")


declining_with_demand = (
    (filtered["trend_direction"] == "down") & (filtered["impressions_90d"] >= 100)
).sum()
print(f"\nRows matching 'declining_with_demand' reason code: {declining_with_demand:,}")


stale_visible = (
    (filtered["days_since_last_update"] >= 180) & (filtered["impressions_90d"] >= 500)
).sum()
print(f"\nRows matching 'stale_visible_page' reason code: {stale_visible:,}")

Total rows in raw dataset: 30,000
Rows after filter (impressions_90d > 0 and content_age_days >= 90): 30,000

Rows with trend_direction == 'down': 16,262 (54.2%)

Rows matching 'declining_with_demand' reason code: 13,152

Rows matching 'stale_visible_page' reason code: 17


## 4. Careful words: what I can and can't claim

This is a decision-support project, not a causal one. I can rank pages by evidence of risk or opportunity so a human reviewer spends limited time well. I cannot claim that refreshing a flagged page will cause it to recover, since that would require an experiment or causal design this data doesn't support. The label I'm using (trend_direction == "down") is a proxy for decline, not verified ground truth, and it's calculated from the current window rather than a future outcome. A stronger version of this project would predict a future window instead. I also won't use any FlyRank product computed field (health_score, priority_score, action_type) as a model feature, since that would let the model just learn to copy an existing rule rather than discover anything from the underlying signals. Finally, I make no claims about Google's ranking algorithm, only about patterns observable in this dataset.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.